#Aplicação Estatística a problemas reais da Biologia

## Curvas de Crescimento

O crescimento microbiano segue um padrão sigmoidal bem caracterizado, com fases distintas: **lag**, **exponencial**, **estacionária** e **declínio**. A análise quantitativa destas curvas permite extrair parâmetros biológicos que descrevem o comportamento de cada cultura.

---

### Parâmetros-chave

| Parâmetro | Símbolo | Significado biológico |
|---|---|---|
| Plateau | **A** | Densidade ótica máxima atingida (capacidade de carga) |
| Taxa de crescimento máxima | **μ_max** | Inclinação máxima da curva — quão rápido as células se dividem |
| Fase lag | **λ** | Tempo que a cultura demora a iniciar o crescimento exponencial |

---

### Abordagem 1 — Modelo Não-Linear (O "Padrão Ouro")

Ajustam-se os dados a uma função matemática (e.g. Gompertz modificado) que descreve o crescimento completo. Este ajuste devolve diretamente os três parâmetros acima, permitindo comparações estatísticas entre condições:

- **"O tratamento atrasou o início do crescimento?"** → Comparar **λ**
- **"Qual grupo cresce mais rápido?"** → Comparar **μ_max**
- **"Qual grupo atinge maior densidade?"** → Comparar **A**

### Abordagem 2 — Regressão Linear na Fase Exponencial

Se o interesse é apenas a velocidade de duplicação:

1. Transformar o eixo Y em **log₁₀**.
2. Selecionar apenas a **fase exponencial** (a porção linear do gráfico log).
3. Ajustar uma **Regressão Linear** a cada grupo.
4. Usar o teste de **Comparação de Inclinações** (*Comparison of Slopes*) para avaliar se as taxas de crescimento diferem significativamente.

---

### Nesta aula

Vamos aplicar ambas as abordagens a dados de crescimento de 6 estirpes microbianas, estimando os parâmetros primeiro graficamente e depois numericamente.

In [ ]:
# Importar Pacotes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# FASE 1 — Carregar e visualizar os dados
1. Olhando para os dados conseguem responder às seguintes perguntas: 
   1. **"Qual o micróbio que demora mais tempo a iniciar o crescimento?"**
   2. **"Qual o micróbio que demora mais tempo a iniciar o crescimento?"** → Comparar **λ**
   3. **"Qual o micróbio que consegue atingir a maior densidade?"**  → Comparar **A**
   4. **"Qual o micróbio cresce mais rápido?"** → Comparar **μ_max**

In [ ]:
# Carregar o CSV
df = pd.read_csv("crescimento_microbiano.csv")

print("Primeiras linhas do DataFrame:")
print(df.head(5))

print(df['Strain'].unique())  # Remove espaços em branco

print("\nEstatísticas descritivas:")
print(df.describe().round(3))


# ══════════════════════════════════════════════════════════
# FASE 3 — Calcular parâmetros numericamente (por estirpe)
# ══════════════════════════════════════════════════════════

# Extrair tempos e lista de estirpes
time_cols = [c for c in df.columns if c.startswith("t") and c.endswith("h")]
t = np.array([int(c[1:-1]) for c in time_cols], dtype=float)
estirpes = df["Strain"].unique()

def calcular_parametros(t, od):
    """
    Calcula plateau, lag e mu_max a partir de dados brutos.
    """
    # Plateau — média dos últimos 20% dos pontos
    n_plateau = max(3, int(0.2 * len(od)))
    A = od[-n_plateau:].mean()

    # Derivada numérica (diferenças finitas)
    dod_dt = np.diff(od) / np.diff(t)
    t_mid  = (t[:-1] + t[1:]) / 2

    # mu_max — valor máximo da derivada
    idx_max = np.argmax(dod_dt)
    mu = dod_dt[idx_max]

    # Fase lag — interseção da tangente no ponto de inflexão com OD=0
    t_inf = t_mid[idx_max]
    od_inf = od[idx_max]
    lam = t_inf - od_inf / mu

    return {"A": round(A, 3), "mu": round(mu, 3), "lam": round(lam, 3)}


# Aplicar a todas as estirpes (usando a média das réplicas)
print("\nParâmetros calculados numericamente (média das réplicas):")
print(f"{'Estirpe':<25} {'Plateau (A)':>12} {'μ_max':>10} {'Lag (h)':>10}")
print("─" * 60)

resultados = {}
for estirpe in estirpes:
    od_mean = df[df["Strain"] == estirpe][time_cols].mean().values
    params = calcular_parametros(t, od_mean)
    resultados[estirpe] = params
    print(f"{estirpe:<25} {params['A']:>12.3f} {params['mu']:>10.3f} {params['lam']:>10.3f}")

# Guardar num DataFrame
df_params = pd.DataFrame(resultados).T
df_params.index.name = "Estirpe"

# ── Gráfico comparativo dos parâmetros ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
params_lista = ["A", "mu", "lam"]
titulos = ["Plateau (A)", "Taxa máxima μ (OD/h)", "Fase lag (h)"]

for ax, param, titulo in zip(axes, params_lista, titulos):
    valores = df_params[param]
    ax.bar(range(len(estirpes)), valores)
    ax.set_xticks(range(len(estirpes)))
    ax.set_xticklabels([e.replace("_", "\n") for e in estirpes], fontsize=7)
    ax.set_title(titulo)
    ax.set_ylabel(param)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Comparação de parâmetros de crescimento", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

1. Filtrar a dataframe para os valores a utilizar

In [ ]:
estirpe = "E_coli_K12" # defina a estirpe a analisar

# Filtrar réplicas do microorganimo escolhido
df_estirpe = df[df["Strain"] == estirpe]

# Filtrar a dataframe para as colunas com dados apenas
time_cols = [c for c in df.columns if c.startswith("t") and c.endswith("h")] # criar lista de colunas de tempo
df_estirpe_time = df_estirpe[time_cols] # criar dataframe apenas com as colunas de tempo para a estirpe escolhida


2. Estimar o valor médio e desvio padrão das diferentes réplicas e criar arrays com os dados.

In [ ]:
od_mean = df_estirpe_time.mean().values # calcular média para cada tempo
print(f"Médias calculadas por cada periodo de tempo analisado:\n{od_mean}")
od_std  = df_estirpe_time.std().values # calcular desvio-padrão para cada tempo
print(f"Desvios-padrão:\n{od_std}")

3. Criar array dos valores X - tempo:

In [ ]:

# Extrair os tempos a partir dos nomes das colunas (t0h, t1h, …)
columns = df.columns.to_list() # lista o nome das colunas do dataframe
time_cols = columns[2:]  # mantém apenas as colunas de tempo (assumindo que as 2 primeiras são 'Strain' e 'Replicate')
print(f"Tempos analisados: {time_cols}")
t = np.array([int(c[1:-1]) for c in time_cols])
print(f"Array de tempos (horas): {t}")




fig, ax = plt.subplots(figsize=(8, 5))

# Curva média com banda de desvio-padrão
ax.plot(t, od_mean, "o-", color="#3266ad", linewidth=2, markersize=5, label=f"{estirpe} (média, n={len(df_estirpe)})")
ax.fill_between(t, od_mean - od_std, od_mean + od_std, color="#3266ad", alpha=0.2, label="± SD")

# ── Plateau (A) ─────────────────────────────────────────
A_estimado = od_mean[-5:].mean()
ax.axhline(A_estimado, color="gray", linestyle="--", label=f"Plateau ≈ {A_estimado:.2f}")

# ── Fase lag (lam) ──────────────────────────────────────
# Os alunos identificam visualmente e registam o tempo
lam_estimado = 0.5   # horas — alterar consoante a observação
ax.axvline(lam_estimado, color="orange", linestyle="--", label=f"Lag ≈ {lam_estimado}h")

# ── Taxa máxima (mu) ─────────────────────────────────────
dod_dt = np.diff(od_mean) / np.diff(t)
idx_max = np.argmax(dod_dt)
mu_estimado = dod_dt[idx_max]
t_inflexao = (t[idx_max] + t[idx_max + 1]) / 2

ax.annotate(f"μ_max ≈ {mu_estimado:.2f} OD/h",
            xy=(t_inflexao, od_mean[idx_max]),
            xytext=(t_inflexao + 3, od_mean[idx_max] - 0.5),
            arrowprops=dict(arrowstyle="->", color="red"),
            color="red", fontsize=10)

ax.set_xlabel("Tempo (horas)")
ax.set_ylabel("OD600")
ax.set_title(f"{estirpe} — estimativa gráfica dos parâmetros")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n{'─'*40}")
print(f"Estimativas gráficas para {estirpe}:")
print(f"  Plateau (A)  ≈ {A_estimado:.3f}")
print(f"  Fase lag     ≈ {lam_estimado:.1f} h")
print(f"  μ_max        ≈ {mu_estimado:.3f} OD/h")
print(f"{'─'*40}")

In [ ]:
#per replica

def calcular_parametros(t, od):
    """
    Calcula plateau, lag e mu_max a partir de dados brutos.

    Parâmetros
    ----------
    t   : array de tempos (horas)
    od  : array de densidades óticas (OD600)

    Devolve
    -------
    dict com A (plateau), lam (lag), mu (taxa máx.)
    """
    # Plateau — média dos últimos 20% dos pontos
    n_plateau = max(3, int(0.2 * len(od)))
    A = od[-n_plateau:].mean()

    # Derivada numérica (diferenças finitas)
    dod_dt = np.diff(od) / np.diff(t)
    t_mid  = (t[:-1] + t[1:]) / 2

    # mu_max — valor máximo da derivada
    idx_max = np.argmax(dod_dt)
    mu = dod_dt[idx_max]

    # Fase lag — interseção da tangente no ponto de inflexão com OD=0
    t_inf = t_mid[idx_max]
    od_inf = od[idx_max]
    lam = t_inf - od_inf / mu       # equação da reta tangente

    return {"A": round(A, 3), "mu": round(mu, 3), "lam": round(lam, 3)}


# Aplicar a todas as estirpes
print("\nParâmetros calculados numericamente:")
print(f"{'Estirpe':<18} {'Plateau (A)':>12} {'μ_max':>10} {'Lag (h)':>10}")
print("─" * 54)

resultados = {}
for estirpe in estirpes:
    od = df[estirpe].values
    params = calcular_parametros(t, od)
    resultados[estirpe] = params
    print(f"{estirpe:<18} {params['A']:>12.3f} {params['mu']:>10.3f} {params['lam']:>10.3f}")

# Guardar num DataFrame
df_params = pd.DataFrame(resultados).T
df_params.index.name = "Estirpe"


In [ ]:

# ── Gráfico comparativo dos parâmetros ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
params_lista = ["A", "mu", "lam"]
titulos = ["Plateau (A)", "Taxa máxima μ (OD/h)", "Fase lag (h)"]

for ax, param, titulo in zip(axes, params_lista, titulos):
    valores = df_params[param]
    barras = ax.bar(range(len(estirpes)), valores)
    ax.set_xticks(range(len(estirpes)))
    ax.set_xticklabels([e.replace("_", "\n") for e in estirpes], fontsize=8)
    ax.set_title(titulo)
    ax.set_ylabel(param)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Comparação de parâmetros de crescimento", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#gerar curvas de todos a partir dos dados dados pela turma
# ──────────────────────────────────────────────
# 3. GERAR OS DADOS — NumPy
# ──────────────────────────────────────────────
t = np.arange(0, 48, 1)          # 0 a 47 horas

np.random.seed(42)                # reprodutibilidade

data = {"Tempo_h": t}

for nome, p in strains.items():
    od = gompertz(t, p["A"], p["mu"], p["lam"])
    ruido = np.random.normal(0, 0.02, size=len(t))   # ruído experimental
    data[nome] = np.clip(od + ruido, 0, None).round(3)

# ──────────────────────────────────────────────
# 4. CRIAR DATAFRAME — Pandas
# ──────────────────────────────────────────────
df = pd.DataFrame(data)

print(df.head(10))
print("\nEstatísticas descritivas:")
print(df.describe().round(3))

# Exportar para CSV (opcional)
# df.to_csv("crescimento_microbiano.csv", index=False)

# ──────────────────────────────────────────────
# 5. VISUALIZAR — Matplotlib
# ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

cores = ["#3266ad", "#e07b39", "#3a9c5f", "#c94f7c", "#8e5db5", "#73726c"]

for (nome, _), cor in zip(strains.items(), cores):
    ax.plot(df["Tempo_h"], df[nome], label=nome, color=cor, linewidth=2)

ax.set_xlabel("Tempo (horas)", fontsize=12)
ax.set_ylabel("Densidade ótica (OD600)", fontsize=12)
ax.set_title("Curvas de crescimento microbiano — 6 estirpes", fontsize=14)
ax.legend(loc="lower right", framealpha=0.9)
ax.set_xlim(0, 47)
ax.set_ylim(0, 3.2)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Expressão RT-PCR

*   Normalização (2−ΔΔCt​)
*   t-test
*   ANOVA One-way 3 grupos

## Contagem de Colônias e Estatística

Exemplo de marcação de bactérias contaminantes de água com colorimetros

*  Cáluclo de CFU's, média de diferentes diluições


*   Verificar se os dados são paramétricos.
    *   t-test
    *   **ou**
    *   Mann-Whitney

### Representação Gráfica
Para publicações científicas, a convenção é:

*   **Eixo Y:** Em escala logarítmica (log10​).
*   **Barras de Erro:** Use o Erro Padrão da Média (SEM) ou Desvio Padrão (SD).
*   **Asteriscos:** Indique a significância (∗p<0,05, ∗∗p<0,01).